# DeepLabV3+ seed=3407 仅测试

加载已有最佳 checkpoint，只评估原始 test 集，不重新训练。运行前开启 GPU、Internet，并添加 v5 数据集与 DeepLab seed3407 checkpoint 数据集。

In [ ]:
DATA_ROOT = '/kaggle/input/datasets/yuanssy/v5data/datasetv5_random811'
GIT_URL = 'https://github.com/song110585-cpu/lunar-linear.git'
GIT_REF = 'test-new-module'
CHECKPOINT = None  # 自动查找路径中同时包含 deeplab 和 3407 的 best_model 文件

In [ ]:
import torch
print(f'PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}')
assert torch.cuda.is_available(), '请先开启 Kaggle GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
!pip install rasterio segmentation-models-pytorch -q

In [ ]:
import os, subprocess
repo_root = '/kaggle/working/lunar-linear-eval3407'
if not os.path.isdir(os.path.join(repo_root, '.git')):
    subprocess.run(['git', 'clone', GIT_URL, repo_root], check=True)
subprocess.run(['git', 'checkout', GIT_REF], cwd=repo_root, check=True)
subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=repo_root, check=True)

In [ ]:
import glob, os
assert os.path.isdir(DATA_ROOT), f'数据集不存在: {DATA_ROOT}'
if CHECKPOINT is None:
    all_models = glob.glob('/kaggle/input/**/best_model*', recursive=True)
    candidates = [p for p in all_models if 'deeplab' in p.lower() and '3407' in p.lower()]
    print('发现的 checkpoint:', *all_models, sep='\n  ')
    assert len(candidates) == 1, '无法唯一确定 DeepLab seed3407 权重，请在配置格手动填写 CHECKPOINT'
    CHECKPOINT = candidates[0]
assert os.path.isfile(CHECKPOINT), f'checkpoint 不存在: {CHECKPOINT}'
print('使用:', CHECKPOINT)

In [ ]:
import os, sys, subprocess
repo = os.path.join(repo_root, 'LTL-Net')
result_dir = '/kaggle/working/result_DeepLabV3Plus_resnet50_seed3407_eval'
os.makedirs(result_dir, exist_ok=True)
log_path = os.path.join(result_dir, 'eval_log.txt')
cmd = [sys.executable, 'scripts/train_baseline.py',
       '--model', 'DeepLabV3Plus', '--encoder', 'resnet50',
       '--data-dir', DATA_ROOT, '--seed', '3407',
       '--run-name', 'DeepLabV3Plus_resnet50_seed3407_eval',
       '--eval-only', '--checkpoint', CHECKPOINT]
print('RUN:', ' '.join(cmd), flush=True)
with open(log_path, 'w', encoding='utf-8') as log_file:
    process = subprocess.Popen(cmd, cwd=repo, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'评估失败: return code={return_code}')

In [ ]:
import json, shutil, os
metrics_path = os.path.join(result_dir, 'metrics.json')
with open(metrics_path, encoding='utf-8') as f:
    result = json.load(f)
print(json.dumps(result['test'], ensure_ascii=False, indent=2))
archive = shutil.make_archive(result_dir, 'zip', '/kaggle/working', os.path.basename(result_dir))
print('下载:', archive, f'{os.path.getsize(archive)/1024/1024:.1f} MB')